# AxonScope Colab Double-Cable Linear Solvers

Use this notebook in a Google Colab GPU runtime. It clones the moving `bench-colab` branch, installs AxonScope, runs the exact double-cable 2x2 block-tridiagonal linear-solver benchmark, optionally runs a forced CPU comparison, writes CSV/JSON summaries, zips the run folder, and downloads it through the browser.

Local setup before running this notebook:

```bash
git add -A
git commit -m "Benchmark double-cable linear solvers"
make bench-colab-push
```

After download, unzip the archive into your local `benchmark/results/solvers/` folder.

In [ ]:
import csv
import datetime
import json
import os
import pathlib
import shutil
import subprocess

from google.colab import files

# Replace only if you are testing a fork.
REPO_URL = "https://github.com/louisreg/AxonScope.git"
BRANCH = "bench-colab"

# Cases:
# - smoke: tiny sanity run.
# - gpu_matrix: useful first GPU sweep for Phase 7.6.3 decisions.
# - gpu_full: roadmap-sized sweep; can take longer and produce many compiles.
# - trace_pcr_adaptive: focused profiler run for one adaptive PCR case.
CASE = "gpu_matrix"

RUN_GPU = True
RUN_CPU = False
TRACE_GPU = True
TRACE_CPU = False
CREATE_PERFETTO = False

CASES = {
    "smoke": {
        "batch_sizes": [8],
        "nx": [16],
        "dtypes": ["float32"],
        "solvers": ["thomas", "pcr_soa"],
        "warmups": 0,
        "repeats": 1,
        "skip_reference": False,
    },
    "gpu_matrix": {
        "batch_sizes": [128, 512, 1024, 2048, 4096],
        "nx": [32, 51, 64, 96],
        "dtypes": ["float32"],
        "solvers": ["thomas", "pcr", "pcr_soa", "pcr_adaptive"],
        "warmups": 1,
        "repeats": 5,
        "skip_reference": False,
    },
    "gpu_full": {
        "batch_sizes": [1, 8, 128, 512, 1024, 2048, 4096],
        "nx": [16, 32, 51, 64, 96, 100, 128],
        "dtypes": ["float32", "float64"],
        "solvers": ["thomas", "pcr", "pcr_soa", "pcr_adaptive"],
        "warmups": 1,
        "repeats": 5,
        "skip_reference": False,
    },
    "trace_pcr_adaptive": {
        "batch_sizes": [1024],
        "nx": [64],
        "dtypes": ["float32"],
        "solvers": ["pcr_adaptive"],
        "warmups": 1,
        "repeats": 1,
        "skip_reference": False,
        "trace_gpu": True,
    },
}

PKG_DIR = pathlib.Path("/content/AxonScope")
RESULTS_ROOT = PKG_DIR / "benchmark/results/solvers"


def run_command(command, *, cwd=None, env=None):
    print("\n$", " ".join(str(part) for part in command), flush=True)
    subprocess.run(command, cwd=cwd, env=env, check=True)


def setup_repo():
    if PKG_DIR.exists():
        shutil.rmtree(PKG_DIR)
    run_command(["git", "clone", "--depth", "1", "--branch", BRANCH, REPO_URL, str(PKG_DIR)])
    run_command(["git", "rev-parse", "--short", "HEAD"], cwd=PKG_DIR)
    run_command(["python", "-m", "pip", "install", "-U", "pip"])
    run_command(["python", "-m", "pip", "install", "-e", ".[benchmark]"], cwd=PKG_DIR)
    run_command(["bash", "-lc", "nvidia-smi || true"], cwd=PKG_DIR)


def verify_backend(label, *, env=None):
    expected = "gpu" if label == "gpu" else "cpu"
    code = f"""
import jax
backend = jax.default_backend()
print('jax backend:', backend)
print('jax devices:', jax.devices())
if backend != {expected!r}:
    raise SystemExit(f'Expected JAX backend {expected!r}, got {{backend!r}}.')
"""
    run_command(["python", "-c", code], cwd=PKG_DIR, env=env)


def solver_command(config, *, label, out_dir, trace=False):
    command = [
        "python",
        "benchmark/solvers/bench_double_cable_linear_solvers.py",
        "--batch-sizes",
        *[str(value) for value in config["batch_sizes"]],
        "--nx",
        *[str(value) for value in config["nx"]],
        "--dtypes",
        *config["dtypes"],
        "--solvers",
        *config["solvers"],
        "--warmups",
        str(config["warmups"]),
        "--repeats",
        str(config["repeats"]),
        "--out-dir",
        str(out_dir),
        "--prefix",
        label,
    ]
    if config.get("skip_reference", False):
        command.append("--skip-reference")
    if trace:
        command.extend(["--jax-trace", "--jax-trace-dir", str(out_dir / label / "jax_traces")])
        if CREATE_PERFETTO:
            command.append("--jax-trace-create-perfetto")
    return command


def run_solver_label(label, config, *, out_dir, env=None, trace=False):
    verify_backend(label, env=env)
    run_command(solver_command(config, label=label, out_dir=out_dir, trace=trace), cwd=PKG_DIR, env=env)
    return out_dir / label / "summary.csv"


def load_summary(path):
    with path.open(newline="", encoding="utf-8") as handle:
        return list(csv.DictReader(handle))


def row_key(row):
    return (
        row["requested_solver"],
        row["resolved_solver"],
        row["kernel_solver"],
        row["batch_size"],
        row["nx"],
        row["dtype"],
    )


def write_comparison(out_dir, gpu_csv, cpu_csv):
    gpu_rows = {row_key(row): row for row in load_summary(gpu_csv)}
    cpu_rows = {row_key(row): row for row in load_summary(cpu_csv)}
    rows = []
    for key in sorted(gpu_rows.keys() & cpu_rows.keys()):
        gpu = gpu_rows[key]
        cpu = cpu_rows[key]
        gpu_ms = float(gpu["steady_median_ms"])
        cpu_ms = float(cpu["steady_median_ms"])
        rows.append({
            "requested_solver": key[0],
            "resolved_solver": key[1],
            "kernel_solver": key[2],
            "batch_size": key[3],
            "nx": key[4],
            "dtype": key[5],
            "gpu_median_ms": gpu_ms,
            "cpu_median_ms": cpu_ms,
            "cpu_over_gpu_speedup_x": cpu_ms / gpu_ms if gpu_ms else "",
            "gpu_node_solves_per_s": gpu["node_solves_per_s"],
            "cpu_node_solves_per_s": cpu["node_solves_per_s"],
            "gpu_max_abs_error_vs_thomas64": gpu["max_abs_error_vs_thomas64"],
            "cpu_max_abs_error_vs_thomas64": cpu["max_abs_error_vs_thomas64"],
        })
    path = out_dir / "comparison_summary.csv"
    with path.open("w", newline="", encoding="utf-8") as handle:
        writer = csv.DictWriter(handle, fieldnames=list(rows[0]) if rows else [])
        if rows:
            writer.writeheader()
            writer.writerows(rows)
    return path


def print_top_summary(label, csv_path, *, top_n=12):
    rows = load_summary(csv_path)
    print(f"\n{label.upper()} summary: {csv_path}")
    for row in rows[:top_n]:
        print(
            f"{row['requested_solver']}({row['kernel_solver']}) "
            f"B={row['batch_size']} Nx={row['nx']} {row['dtype']}: "
            f"median {float(row['steady_median_ms']):.3f} ms | "
            f"nodes/s {float(row['node_solves_per_s']):.3e} | "
            f"err {row['max_abs_error_vs_thomas64']}"
        )
    if len(rows) > top_n:
        print(f"... {len(rows) - top_n} more rows")


setup_repo()

if CASE not in CASES:
    raise ValueError(f"Unknown CASE={CASE!r}; choose one of {sorted(CASES)}")
config = dict(CASES[CASE])
run_id = f"colab_double_cable_linear_{CASE}_{datetime.datetime.now().strftime('%Y%m%d_%H%M%S')}"
out_dir = RESULTS_ROOT / run_id
out_dir.mkdir(parents=True, exist_ok=True)

print("Selected solver benchmark case:", CASE)
print(json.dumps(config, indent=2, sort_keys=True))

gpu_csv = None
cpu_csv = None

if RUN_GPU:
    gpu_trace = TRACE_GPU or bool(config.get("trace_gpu", False))
    gpu_csv = run_solver_label("gpu", config, out_dir=out_dir, trace=gpu_trace)
    print_top_summary("gpu", gpu_csv)

if RUN_CPU:
    cpu_env = os.environ.copy()
    cpu_env["JAX_PLATFORMS"] = "cpu"
    cpu_trace = TRACE_CPU or bool(config.get("trace_cpu", False))
    cpu_csv = run_solver_label("cpu", config, out_dir=out_dir, env=cpu_env, trace=cpu_trace)
    print_top_summary("cpu", cpu_csv)

comparison_csv = None
if gpu_csv is not None and cpu_csv is not None:
    comparison_csv = write_comparison(out_dir, gpu_csv, cpu_csv)
    print(f"\nCPU/GPU comparison CSV: {comparison_csv}")

archive_path = shutil.make_archive(str(out_dir), "zip", out_dir)
print(f"\nResults folder: {out_dir}")
print(f"Archive: {archive_path}")
files.download(archive_path)


: 